exploration des méthodes possibles de calcul pour simuler les combats OGamiens

Rappel des règles de combats
- 6 tours max
- tir "simultanés" (mais y'a quand même un ordre)
- à chaque tir reçu, on applique la valeur d'attaque au bouclier (arrondi inf en % de la valeur du bouclier total) puis à la coque
    - de ce calcul arrondi découle la règle spéciale d'annulation des dégat si la valeur d'attaque vaut moins de 1% de la valeur des boucliers
- la coque = 10% des points de structure !!
- dès que la coque est endommagée à plus de 30%, le même % est appliqué (à chaque tir reçu) pour savoir si le vaisseau explose
- un vaisseau "détruit" (pendant le calcul d'un tour) n'est retiré qu'à la fin du tour et peut donc continuer à être ciblé ET tirer
- à la fin d'un tour, on retire les vaisseaux détruits et on recharge intégralement tous les boucliers
- à la fin des 6 tours s'il reste des vaisseaux/défenses dans les 2 camps => match nul

### Rapid fire
si un vaisseau tire sur une structure contre laquelle il a du rapid-fire, on lance un dé de la valeur du RF (eg croiseur RF 6 contre Ch lé). Il y a alors 1 chance sur RF (1/6) de ne PAS tirer à nouveau donc 1 - 1/RF d'effectuer un nouveau tir, le cycle continue jusqu'à ne plus proc le RF

## imports

In [2]:
import random
import numpy as np
import pandas as pd
from pathlib import Path

## lecture données

In [ ]:
data_vaisseaux_path = Path("../data/vaisseaux.csv")
df_vaisseaux = pd.read_csv(data_vaisseaux_path)
df_vaisseaux

pour convetir le csv en json "the lazy way"

In [ ]:
df_vaisseaux.to_json(Path("../data/vaisseaux.json"), indent=4, force_ascii=False, orient="records")

*edit temporaire des data en json pour avoir des calculs intéressants. Attention à ne pas l'écraser et le lire ici plutôt que le CSV*

In [17]:
df_vaisseaux = pd.read_json(Path("../data/vaisseaux.json"))

## ajustements technologiques
lecture d'un... yaml ? ouais disons yaml qui contiendra toutes nos paramètres (univers, technologies)

Ajustera les stats des vaisseaux selon les formules

In [10]:
from yaml import safe_load

with Path("../data/universe_parameters.yaml").open("r") as f:
    universe_params = safe_load(f)

universe_params["Technologies"]

{'Réacteur à Combustion': 17,
 'Réacteur à Impultion': 12,
 'Propultion Hyperespace': 10,
 'Technologie Hyperespace': 12,
 'Technologie Armes': 17,
 'Technologie Bouclier': 16,
 'Technologie Protection des vaisseaux spatiaux': 17}

In [18]:
df_vaisseaux.attaque *= 1 + universe_params["Technologies"]["Technologie Armes"] / 10
df_vaisseaux.bouclier *= 1 + universe_params["Technologies"]["Technologie Bouclier"] / 10
df_vaisseaux.structure *= 1 + universe_params["Technologies"]["Technologie Protection des vaisseaux spatiaux"] / 10

In [19]:
df_vaisseaux

,nom,structure,bouclier,attaque,vitesse,fret,consommation
0,Ch lé,10800.0,26.0,135.0,12500,50,20
1,Ch lo,27000.0,65.0,405.0,10000,100,75
2,Croiseur,72900.0,130.0,1080.0,15000,800,300
3,Des,297000.0,1820.0,5400.0,5000,2000,1000
4,Petit transporteur,10800.0,26.0,13.5,10000,5000,20


## création d'une flotte

l'idée serait de constituer un array numpy (on peut surement commencer par un df pandas...)  
afin de permettre ensuite le calcul vectoriel pour chaque individu de la flotte (le RF fera surement chier...)

In [122]:
from data_utils import replicate_rows

df_attaquants = replicate_rows(df_vaisseaux, [1, 50, 0, 0, 0])
df_defenseurs = replicate_rows(df_vaisseaux, [0, 0, 0, 1, 0])

In [123]:
df_attaquants.reset_index(names="tireur", inplace=True)

In [124]:
df_attaquants['cible'] = np.random.randint(0, df_defenseurs.index.max()+1, size=len(df_attaquants))
df_attaquants

,tireur,nom,structure,bouclier,attaque,vitesse,fret,consommation,cible
0,0,Ch lé,4000,10,50,12500,50,20,0
1,1,Ch lo,10000,25,150,10000,100,75,0
2,2,Ch lo,10000,25,150,10000,100,75,0
3,3,Ch lo,10000,25,150,10000,100,75,0
4,4,Ch lo,10000,25,150,10000,100,75,0
5,5,Ch lo,10000,25,150,10000,100,75,0
6,6,Ch lo,10000,25,150,10000,100,75,0
7,7,Ch lo,10000,25,150,10000,100,75,0
8,8,Ch lo,10000,25,150,10000,100,75,0
9,9,Ch lo,10000,25,150,10000,100,75,0


In [104]:
df_attaquants['cible'] = ((df_attaquants.nom == "Ch lo") + (df_attaquants.tireur <= 4)).astype(int)
df_attaquants

,tireur,nom,structure,bouclier,attaque,vitesse,fret,consommation,cible
0,0,Ch lé,4000,10,50,12500,50,20,1
1,1,Ch lo,10000,25,150,10000,100,75,1
2,2,Ch lo,10000,25,150,10000,100,75,1
3,3,Ch lo,10000,25,150,10000,100,75,1
4,4,Ch lo,10000,25,150,10000,100,75,1
5,5,Ch lo,10000,25,150,10000,100,75,1
6,6,Ch lo,10000,25,150,10000,100,75,1
7,7,Ch lo,10000,25,150,10000,100,75,1
8,8,Ch lo,10000,25,150,10000,100,75,1
9,9,Ch lo,10000,25,150,10000,100,75,1


In [125]:
df_defenseurs

,nom,structure,bouclier,attaque,vitesse,fret,consommation
0,Des,110000,700,2000,5000,2000,1000


In [126]:
df_tirs = df_attaquants.merge(
    right=df_defenseurs, 
    how='left', 
    left_on='cible', 
    right_index=True,
    suffixes=["_A", "_D"]
)

### extrait du forum, sur le calcul des boucliers
https://board.fr.ogame.gameforge.com/index.php?thread/728567-casser-un-grand-bouclier/&postID=12026202#post12026202

Dans le cas ou le tir a une attaque inférieur au bouclier restant, le tir atteint le bouclier avec une valeur égale à l'arrondi au pourcent inférieur de la valeur maximale du bouclier (si c'est pas très clair, voir l'exemple ligne suivante qui sera plus parlant).

Dans ton cas, un croiseur attaque 720 sur un bouclier qui a une valeur max de 16000 : 720 / 16000 = 4.5%, arrondi inférieur = 4%, donc le bouclier va diminuer de 4% de sa valeur totale, soit 640.

Le tir de 720 est entièrement absorbé, mais ne diminue le bouclier que de 640.

C'est ce qui explique par ailleurs qu'un tir inférieur à 1% du bouclier ne tape pas du tout, vu qu'il est arrondi à 0% (et dans ton cas, les GT ne tapent donc pas le bouclier vu que c'est 0.06% du bouclier).

In [127]:
df_tirs["dmg_shield"] = (np.floor(100 * df_tirs.attaque_A / df_tirs.bouclier_D) * df_tirs.bouclier_D // 100).astype(int)
df_tirs = df_tirs.sort_values(by=["cible", "tireur"])[["tireur", "nom_A", "attaque_A", "cible", "nom_D", "bouclier_D", "structure_D", "dmg_shield"]]
# df_tirs

il y a un truc à faire avec un tri par cible puis attaquant, ce qui permet d'appliquer une somme cumulative conditionnelle ensuite et résoudre les tirs sur une opération simplifiée (somme et soustraction, pas de produit/division)

In [128]:
# dégats cumulés par cible dans l'ordre de l'id des tireurs
df_tirs["shield_aft_hit"] = df_tirs.bouclier_D - df_tirs.groupby("cible")["dmg_shield"].cumsum()

# colonne décalée pour avoir la valeur du shield avant le tir
df_tirs["shield_bef_hit"] = df_tirs.groupby("cible")["shield_aft_hit"].shift(1).fillna(df_tirs.bouclier_D).astype(int)

# vérifie si le tir dépasse la valeur max des boucliers
# ie. si les boucliers sont HS après le tir
# df_tirs["is_overkill"] = df_tirs.shield_aft_hit <= 0
is_overkill = df_tirs.shield_aft_hit <= 0 # on externalise le filtre booléen (pour alléger le df)

# vérifie si les boucliers étaient déjà HS avant le tir
# df_tirs["déjà_hs"] = df_tirs.shield_bef_hit <= 0
déjà_hs = df_tirs.shield_bef_hit <= 0

df_tirs

,tireur,nom_A,attaque_A,cible,nom_D,bouclier_D,structure_D,dmg_shield,shield_aft_hit,shield_bef_hit
0,0,Ch lé,50,0,Des,700,110000,49,651,700
1,1,Ch lo,150,0,Des,700,110000,147,504,651
2,2,Ch lo,150,0,Des,700,110000,147,357,504
3,3,Ch lo,150,0,Des,700,110000,147,210,357
4,4,Ch lo,150,0,Des,700,110000,147,63,210
5,5,Ch lo,150,0,Des,700,110000,147,-84,63
6,6,Ch lo,150,0,Des,700,110000,147,-231,-84
7,7,Ch lo,150,0,Des,700,110000,147,-378,-231
8,8,Ch lo,150,0,Des,700,110000,147,-525,-378
9,9,Ch lo,150,0,Des,700,110000,147,-672,-525


### calcul des dégats infligés à la coque
le **breakpoint** est le moment où le tir est **overkill** mais les boucliers ne sont **pas** déjà HS avant le tir

In [129]:
# filtre (1ère passe) pour attribuer les dégats complets (ie. la totalité des dégats) infligés à la coque
hull_dmg = np.where(déjà_hs, df_tirs.attaque_A, 0)
hull_dmg

array([  0,   0,   0,   0,   0,   0, 150, 150, 150, 150, 150, 150, 150,
       150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150,
       150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150,
       150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150])

In [130]:
# 2ème passe pour la situation de breakpoint
df_tirs["hull_dmg"] = np.where(
    (is_overkill) & (~déjà_hs), # filtre uniquement pour le breakpoint
    np.maximum(0, df_tirs.attaque_A - df_tirs.shield_bef_hit), # calcule la valeur d'attaque restante après les dégats aux boucliers
    hull_dmg
)
df_tirs["hull_dmg_cum"] = df_tirs.groupby("cible")["hull_dmg"].cumsum()
df_tirs

,tireur,nom_A,attaque_A,cible,nom_D,bouclier_D,structure_D,dmg_shield,shield_aft_hit,shield_bef_hit,hull_dmg,hull_dmg_cum
0,0,Ch lé,50,0,Des,700,110000,49,651,700,0,0
1,1,Ch lo,150,0,Des,700,110000,147,504,651,0,0
2,2,Ch lo,150,0,Des,700,110000,147,357,504,0,0
3,3,Ch lo,150,0,Des,700,110000,147,210,357,0,0
4,4,Ch lo,150,0,Des,700,110000,147,63,210,0,0
5,5,Ch lo,150,0,Des,700,110000,147,-84,63,87,87
6,6,Ch lo,150,0,Des,700,110000,147,-231,-84,150,237
7,7,Ch lo,150,0,Des,700,110000,147,-378,-231,150,387
8,8,Ch lo,150,0,Des,700,110000,147,-525,-378,150,537
9,9,Ch lo,150,0,Des,700,110000,147,-672,-525,150,687


calcul du % d'endommagement de la coque  
Rappels : 
+ au delà de 30% de dégats, % de destruction = % dégats
+ coque = 10% des points de structure (= coût M + coût C, btw)


In [148]:
hull_dmg_percent = 10 * df_tirs.hull_dmg_cum / df_tirs.structure_D
df_tirs["destruction_chance"] = np.where(
    hull_dmg_percent > 0.3,
    hull_dmg_percent,
    0
)
destruction_roll = np.random.random(size=len(df_tirs))
df_tirs["is_destroyed"] = destruction_roll < df_tirs["destruction_chance"]
df_tirs

,tireur,nom_A,attaque_A,cible,nom_D,bouclier_D,structure_D,dmg_shield,shield_aft_hit,shield_bef_hit,hull_dmg,hull_dmg_cum,destruction_chance,is_destroyed
0,0,Ch lé,50,0,Des,700,110000,49,651,700,0,0,0.000000,False
1,1,Ch lo,150,0,Des,700,110000,147,504,651,0,0,0.000000,False
2,2,Ch lo,150,0,Des,700,110000,147,357,504,0,0,0.000000,False
3,3,Ch lo,150,0,Des,700,110000,147,210,357,0,0,0.000000,False
4,4,Ch lo,150,0,Des,700,110000,147,63,210,0,0,0.000000,False
5,5,Ch lo,150,0,Des,700,110000,147,-84,63,87,87,0.000000,False
6,6,Ch lo,150,0,Des,700,110000,147,-231,-84,150,237,0.000000,False
7,7,Ch lo,150,0,Des,700,110000,147,-378,-231,150,387,0.000000,False
8,8,Ch lo,150,0,Des,700,110000,147,-525,-378,150,537,0.000000,False
9,9,Ch lo,150,0,Des,700,110000,147,-672,-525,150,687,0.000000,False
